In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import mne

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:

# Load individual EEG 
DATA_ROOT = Path("/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives")

def load_eeg(subject, session):
    
    path = os.path.join(
        ROOT,
        f"sub-{subject:02d}",
        f"ses-{session:02d}",
        f"sub-{subject:02d}_ses-{session:02d}_eeg-epo.fif"
    )

    epochs = mne.read_epochs(
        path,
        preload=True,
        verbose=False
    )

    return epochs

In [ ]:
eeg_path = list(DATA_ROOT.rglob("*_eeg-epo.fif"))[0]

epochs = mne.read_epochs(
    eeg_path,
    preload=True,
    verbose=False
)

X = epochs.get_data()

print("File:", eeg_path)
print("Shape:", X.shape)
print("Sampling rate:", epochs.info["sfreq"])
print("Channels:", len(epochs.ch_names))
print("Time range:", epochs.times[0], "to", epochs.times[-1])


event_path = Path(
    str(eeg_path).replace("_eeg-epo.fif", "_events.dat")
)

events = pd.read_pickle(event_path)



In [ ]:
X = epochs.get_data()

print("EEG shape:", X.shape)
print("Events shape:", events.shape)

assert len(X) == len(events)

print("Classes:", np.unique(events[:, 1], return_counts=True))
print("Conditions:", np.unique(events[:, 2], return_counts=True))
print("Sessions:", np.unique(events[:, 3], return_counts=True))



In [ ]:
CLASS_NAMES = {
    0: "Up",
    1: "Down",
    2: "Right",
    3: "Left"
}
# condition == 1 means Inner Speech
inner_mask = events[:, 2] == 1

X_inner = X[inner_mask]
y_inner = events[inner_mask, 1].astype(np.int64)

print("All trials:", len(X))
print("Inner-speech trials:", len(X_inner))

for label in np.unique(y_inner):
    print(
        CLASS_NAMES[label],
        np.sum(y_inner == label)
    )



In [ ]:
# Keep the signal 1.0s-3.5s. That's the official action interval.
times = epochs.times

action_mask = (times >= 1.0) & (times < 3.5)

X_inner = X_inner[:, :, action_mask]

print(X_inner.shape)

In [ ]:
import plotly.graph_objects as go

trial = 0
channel = 0

signal = X_inner[trial, channel] * 1e6  # volts -> µV

time_action = times[action_mask]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_action,
        y=signal,
        mode="lines",
        name=epochs.ch_names[channel]
    )
)

fig.update_layout(
    title=f"Inner Speech: {CLASS_NAMES[y_inner[trial]]}",
    xaxis_title="Time (seconds)",
    yaxis_title="Amplitude (µV)"
)

fig.show()

# Exploring the EEG channels on the area of the scalp

In [ ]:
import mne
import numpy as np
import pandas as pd


montage = mne.channels.make_standard_montage("biosemi128")

epochs.set_montage(
    montage,
    match_case=False,
    on_missing="warn"
)



ch_pos = epochs.get_montage().get_positions()["ch_pos"]

rows = []

for ch in epochs.ch_names:

    if ch not in ch_pos:
        continue

    x, y, z = ch_pos[ch]

    rows.append({
        "channel": ch,
        "x": x,   # left (-) / right (+)
        "y": y,   # back (-) / front (+)
        "z": z    # bottom / top
    })

channel_df = pd.DataFrame(rows)

print(channel_df.head())

In [ ]:
# --------------------------------------------------
# 3. Assign rough scalp regions from coordinates
# --------------------------------------------------

def get_region(x, y, z):

    # Front-back classification
    if y > 0.045:
        region = "Frontal"

    elif y > 0.015:
        region = "Front-Central"

    elif y > -0.020:
        region = "Central"

    elif y > -0.050:
        region = "Parietal"

    else:
        region = "Occipital"

    # Strong lateral position → Temporal
    if abs(x) > 0.060 and -0.045 < y < 0.045:
        region = "Temporal"

    return region


channel_df["region"] = channel_df.apply(
    lambda row: get_region(
        row["x"],
        row["y"],
        row["z"]
    ),
    axis=1
)
# --------------------------------------------------
# 4. Assign hemisphere
# --------------------------------------------------

def get_hemisphere(x, threshold=0.005):

    if x < -threshold:
        return "Left"

    elif x > threshold:
        return "Right"

    else:
        return "Midline"


channel_df["hemisphere"] = channel_df["x"].apply(
    get_hemisphere
)

In [ ]:
# --------------------------------------------------
# 5. View channel information
# --------------------------------------------------

channel_df[
    [
        "channel",
        "region",
        "hemisphere",
        "x",
        "y",
        "z"
    ]
].sort_values(
    ["region", "hemisphere"]
)

In [ ]:
# --------------------------------------------------
# 7. Get channels belonging to a specific region
# --------------------------------------------------

frontal_channels = channel_df[
    channel_df["region"] == "Frontal"
]["channel"].tolist()

temporal_channels = channel_df[
    channel_df["region"] == "Temporal"
]["channel"].tolist()

central_channels = channel_df[
    channel_df["region"] == "Central"
]["channel"].tolist()

parietal_channels = channel_df[
    channel_df["region"] == "Parietal"
]["channel"].tolist()

occipital_channels = channel_df[
    channel_df["region"] == "Occipital"
]["channel"].tolist()

print("Frontal:", frontal_channels)
print("Temporal:", temporal_channels)
print("Central:", central_channels)
print("Parietal:", parietal_channels)
print("Occipital:", occipital_channels)

In [ ]:
epochs.plot_sensors(
    kind="3d",
    show_names=True
)

In [ ]:
channel_df.to_csv(
    "biosemi128_channel_regions.csv",
    index=False
)

# Create channels configs for further experiment

In [ ]:
import numpy as np

# --------------------------------------------------
# 1. Get channel positions in the SAME order as X
# --------------------------------------------------

ch_pos = epochs.get_montage().get_positions()["ch_pos"]

channel_names = np.array(epochs.ch_names)

positions = np.array([
    ch_pos[ch]
    for ch in channel_names
])

print(channel_names.shape)
print(positions.shape)

# Expected:
# (128,)
# (128, 3)

In [ ]:
# --------------------------------------------------
# 2. Create spatially distributed channel ranking
#    using Farthest Point Sampling
# --------------------------------------------------

def farthest_point_sampling(positions):

    n_channels = len(positions)

    # Start from electrode closest to top/center of scalp
    center_target = np.array([
        0,
        0,
        positions[:, 2].max()
    ])

    start = np.argmin(
        np.linalg.norm(
            positions - center_target,
            axis=1
        )
    )

    selected = [start]

    remaining = set(
        range(n_channels)
    )

    remaining.remove(start)

    while remaining:

        best_channel = None
        best_distance = -1

        for candidate in remaining:

            distances = np.linalg.norm(
                positions[candidate]
                - positions[selected],
                axis=1
            )

            # Distance to nearest selected electrode
            min_distance = distances.min()

            if min_distance > best_distance:

                best_distance = min_distance
                best_channel = candidate

        selected.append(
            best_channel
        )

        remaining.remove(
            best_channel
        )

    return np.array(selected)
# --------------------------------------------------
# 3. Generate full spatial ranking
# --------------------------------------------------

channel_order = farthest_point_sampling(
    positions
)

print(channel_order)

In [ ]:
# --------------------------------------------------
# 4. Create nested channel configurations
# --------------------------------------------------

channel_configs_idx = {
    128: channel_order[:128],
    64:  channel_order[:64],
    32:  channel_order[:32],
    16:  channel_order[:16]
}

In [ ]:
# --------------------------------------------------
# 5. Convert indices -> channel names
# --------------------------------------------------

channel_configs = {
    n: channel_names[idx].tolist()
    for n, idx in channel_configs_idx.items()
}

for n_channels, channels in channel_configs.items():

    print(
        f"\n{n_channels} CHANNELS"
    )

    print(channels)

In [ ]:
assert set(channel_configs[16]).issubset(
    channel_configs[32]
)

assert set(channel_configs[32]).issubset(
    channel_configs[64]
)

assert set(channel_configs[64]).issubset(
    channel_configs[128]
)

print("All configurations are nested correctly.")

In [ ]:
for n_channels, channels in channel_configs.items():

    subset = channel_df[
        channel_df["channel"].isin(
            channels
        )
    ]

    print(
        f"\n===== {n_channels} channels ====="
    )

    print(
        subset.groupby(
            ["region", "hemisphere"]
        ).size()
    )

In [ ]:
import matplotlib.pyplot as plt
import mne

for n_channels, channels in channel_configs.items():

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    picks = [
        epochs.ch_names.index(ch)
        for ch in channels
    ]

    mne.viz.plot_sensors(
        epochs.info,
        kind="topomap",
        ch_type="eeg",
        show_names=False,
        axes=ax,
        show=False
    )

    xy = mne.channels.layout._find_topomap_coords(
        epochs.info,
        picks=picks
    )

    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=40
    )

    ax.set_title(
        f"{n_channels}-Channel Configuration"
    )

    plt.show()

In [ ]:
X_configs = {
    n: X[:, idx, :]
    for n, idx
    in channel_configs_idx.items()
}

for n, data in X_configs.items():
    print(n, data.shape)

In [ ]:
np.save(
    "channel_configs_idx.npy",
    channel_configs_idx,
    allow_pickle=True
)

np.save(
    "channel_configs_names.npy",
    channel_configs,
    allow_pickle=True
)

# Load all subjects

In [ ]:
all_X = []
all_y = []
all_subjects = []

eeg_files = sorted(
    DATA_ROOT.rglob("*_eeg-epo.fif")
)

print("Number of EEG files:", len(eeg_files))

In [ ]:
for eeg_path in eeg_files:

    # -----------------------
    # Load EEG
    # -----------------------
    epochs = mne.read_epochs(
        eeg_path,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()

    # -----------------------
    # Load events
    # -----------------------
    event_path = Path(
        str(eeg_path).replace(
            "_eeg-epo.fif",
            "_events.dat"
        )
    )

    events = pd.read_pickle(event_path)
    events = np.asarray(events)

    assert len(X) == len(events)

    # -----------------------
    # Subject ID
    # -----------------------
    subject = int(
        re.search(
            r"sub-(\d+)",
            str(eeg_path)
        ).group(1)
    )

    # -----------------------
    # Inner Speech only
    # -----------------------
    inner_mask = events[:, 2] == 1

    X = X[inner_mask]
    y = events[inner_mask, 1].astype(np.int64)

    # -----------------------
    # Action interval
    # -----------------------
    times = epochs.times

    action_mask = (
        (times >= 1.0) &
        (times < 3.5)
    )

    X = X[:, :, action_mask]

    # -----------------------
    # Save
    # -----------------------
    all_X.append(X)
    all_y.append(y)

    all_subjects.append(
        np.full(
            len(X),
            subject,
            dtype=np.int64
        )
    )

In [ ]:
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
subjects = np.concatenate(all_subjects, axis=0)

print("X:", X.shape)
print("y:", y.shape)
print("subjects:", subjects.shape)

In [ ]:
print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())

print("\nClasses:")
for label in range(4):
    print(
        CLASS_NAMES[label],
        np.sum(y == label)
    )

print("\nSubjects:")
for subject in np.unique(subjects):
    print(
        f"Subject {subject}:",
        np.sum(subjects == subject)
    )

# Machine learning approach

In [ ]:
# Split by subject

train_subjects = [1, 2, 3, 4, 5, 6, 7]
val_subjects   = [8]
test_subjects  = [9, 10]

train_mask = np.isin(subjects, train_subjects)
val_mask   = np.isin(subjects, val_subjects)
test_mask  = np.isin(subjects, test_subjects)

X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# Using welch method 
from scipy.signal import welch
def extract_bandpower_features(X, fs):

    bands = {
        "delta": (1, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
        "gamma": (30, 45)
    }

    all_features = []

    for trial in X:

        # trial:
        # [channels, time]

        freqs, psd = welch(
            trial,
            fs=fs,
            nperseg=min(256, trial.shape[-1]),
            axis=-1
        )

        trial_features = []

        for low, high in bands.values():

            mask = (
                (freqs >= low) &
                (freqs < high)
            )

            band_power = psd[:, mask].mean(axis=1)

            # Log-transform power
            band_power = np.log(
                band_power + 1e-12
            )

            trial_features.append(
                band_power
            )

        # [5 bands, 128 channels]
        trial_features = np.array(
            trial_features
        )

        # Flatten:
        # 5 × 128 = 640 features
        trial_features = trial_features.flatten()

        all_features.append(
            trial_features
        )

    return np.array(all_features)

In [ ]:
fs = epochs.info["sfreq"] # Sampling rate

print(fs)

X_train_ml = extract_bandpower_features(
    X_train,
    fs
)

X_val_ml = extract_bandpower_features(
    X_val,
    fs
)

X_test_ml = extract_bandpower_features(
    X_test,
    fs
)

print(X_train_ml.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_ml = scaler.fit_transform(
    X_train_ml
)

X_val_ml = scaler.transform(
    X_val_ml
)

X_test_ml = scaler.transform(
    X_test_ml
)

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm.fit(
    X_train_ml,
    y_train
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_pred = svm.predict(
    X_val_ml
)

print(
    "Val Accuracy:",
    accuracy_score(
        y_val,
        val_pred
    )
)

print(
    "Val Macro F1:",
    f1_score(
        y_val,
        val_pred,
        average="macro"
    )
)
test_pred = svm.predict(
    X_test_ml
)

print(
    "Test Accuracy:",
    accuracy_score(
        y_test,
        test_pred
    )
)

print(
    "Test Macro F1:",
    f1_score(
        y_test,
        test_pred,
        average="macro"
    )
)

# Deep learning baseline


In [ ]:
mean = X_train.mean(
    axis=(0, 2),
    keepdims=True
)

std = X_train.std(
    axis=(0, 2),
    keepdims=True
)

std = np.maximum(
    std,
    1e-8
)

X_train_dl = (
    X_train - mean
) / std

X_val_dl = (
    X_val - mean
) / std

X_test_dl = (
    X_test - mean
) / std

In [ ]:
print(X_train_dl.shape)

print(
    X_train_dl.mean(),
    X_train_dl.std()
)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EEGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )

train_dataset = EEGDataset(
    X_train_dl,
    y_train
)

val_dataset = EEGDataset(
    X_val_dl,
    y_val
)

test_dataset = EEGDataset(
    X_test_dl,
    y_test
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
# EEGNet style baseline
import torch.nn as nn


class EEGNet(nn.Module):

    def __init__(
        self,
        n_channels,
        n_classes=4,
        dropout=0.5
    ):

        super().__init__()

        F1 = 8
        D = 2
        F2 = F1 * D

        # Temporal filtering
        self.temporal = nn.Sequential(

            nn.Conv2d(
                1,
                F1,
                kernel_size=(1, 64),
                padding=(0, 32),
                bias=False
            ),

            nn.BatchNorm2d(F1)
        )

        # Spatial filtering across EEG channels
        self.spatial = nn.Sequential(

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(n_channels, 1),
                groups=F1,
                bias=False
            ),

            nn.BatchNorm2d(F1 * D),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 4)
            ),

            nn.Dropout(dropout)
        )

        # Separable temporal convolution
        self.separable = nn.Sequential(

            nn.Conv2d(
                F2,
                F2,
                kernel_size=(1, 16),
                padding=(0, 8),
                groups=F2,
                bias=False
            ),

            nn.Conv2d(
                F2,
                F2,
                kernel_size=(1, 1),
                bias=False
            ),

            nn.BatchNorm2d(F2),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 8)
            ),

            nn.Dropout(dropout)
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Linear(
            F2,
            n_classes
        )

    def forward(self, x):

        # [B, channels, time]
        # ->
        # [B, 1, channels, time]

        x = x.unsqueeze(1)

        x = self.temporal(x)

        x = self.spatial(x)

        x = self.separable(x)

        x = self.pool(x)

        x = x.flatten(1)

        return self.classifier(x)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = EEGNet(
    n_channels=X_train_dl.shape[1],
    n_classes=4
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
from tqdm import tqdm

def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)

        pred = logits.argmax(dim=1)

        correct += (
            pred == y_batch
        ).sum().item()

        total += len(y_batch)

    return (
        total_loss / total,
        correct / total
    )

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    total = 0

    predictions = []
    labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)

        pred = logits.argmax(dim=1)

        predictions.extend(
            pred.cpu().numpy()
        )

        labels.extend(
            y_batch.cpu().numpy()
        )

    predictions = np.array(predictions)
    labels = np.array(labels)

    accuracy = (
        predictions == labels
    ).mean()

    return (
        total_loss / total,
        accuracy,
        labels,
        predictions
    )

In [ ]:
from tqdm import tqdm

EPOCHS = 50
best_val = 0

pbar = tqdm(
    range(EPOCHS),
    desc="Training EEGNet"
)

for epoch in pbar:

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc, _, _ = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    if val_acc > best_val:

        best_val = val_acc

        torch.save(
            model.state_dict(),
            "best_eegnet.pt"
        )

    pbar.set_postfix(
        train_acc=f"{train_acc:.3f}",
        val_acc=f"{val_acc:.3f}",
        best=f"{best_val:.3f}",
        loss=f"{train_loss:.3f}"
    )

# Controlled three-pipeline channel-reduction experiments

This section turns the existing baselines into repeatable experiments for the fixed **128 / 64 / 32 / 16** channel configurations.

The three tracks are:

1. **Classical ML:** Welch log-bandpower -> training-only StandardScaler -> RBF SVM.
2. **Scratch DL:** raw time-domain EEG -> training-only per-channel normalization -> a newly initialized EEGNet for every channel count.
3. **Pretrained EEG model:** 200 Hz EEG -> training-only per-channel normalization -> frozen spatial interpolation -> pretrained BIOT -> new four-class head, followed by low-learning-rate fine-tuning.

The development protocol remains subjects 1-7 / 8 / 9-10. Test labels are used only once, after selecting the best epoch on subject 8. The helper functions also accept LOSO folds.

BIOT notes:

- BIOT was pretrained at 200 Hz, so only that track is resampled.
- `InterpolatedBIOT` uses the stored BioSemi positions to create a fixed MNE spline projection into BIOT's pretrained montage.
- Loading must succeed from an actual pretrained checkpoint; this code never silently falls back to random initialization.
- Kaggle Internet must be enabled once (or the model/package must be attached as an input). Run `%pip install -q "braindecode[hug]>=1.5,<2"` in a separate cell if needed, then restart the kernel.

References: [BIOT paper](https://papers.neurips.cc/paper_files/paper/2023/file/f6b30f3e2dd9cb53bbf2024402d02295-Paper-Conference.pdf), [BIOT repository](https://github.com/ycq091044/BIOT), and [Braindecode channel-interpolation guide](https://braindecode.org/dev/auto_examples/model_building/plot_channel_interpolation.html).


In [ ]:
import copy
import hashlib
import json
import random
from fractions import Fraction

from scipy.signal import resample_poly, welch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


SEED = 42
CHANNEL_COUNTS = (128, 64, 32, 16)
RESULTS_DIR = Path("results")
CHECKPOINT_DIR = Path("checkpoints")
RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

DEV_SPLIT = {
    "train": (1, 2, 3, 4, 5, 6, 7),
    "val": (8,),
    "test": (9, 10),
}

EXPERIMENT_ROWS = {}
PREDICTION_FRAMES = {}


def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def validate_split(split):
    groups = [set(split[name]) for name in ("train", "val", "test")]
    assert all(groups), "Train, validation, and test subject groups must be non-empty."
    assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
    available = set(np.unique(subjects).tolist())
    assert set.union(*groups).issubset(available)


def split_masks(split):
    validate_split(split)
    return {
        name: np.isin(subjects, split[name])
        for name in ("train", "val", "test")
    }


def channel_config_hash(n_channels):
    payload = json.dumps(channel_configs[n_channels], separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:12]


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def store_predictions(
    protocol, fold, model_name, n_channels, partition, mask, y_pred
):
    key = (protocol, fold, model_name, n_channels, partition)
    PREDICTION_FRAMES[key] = pd.DataFrame({
        "protocol": protocol,
        "fold": fold,
        "model": model_name,
        "n_channels": n_channels,
        "partition": partition,
        "subject": subjects[mask],
        "y_true": y[mask],
        "y_pred": np.asarray(y_pred),
    })


def store_result(
    protocol,
    fold,
    model_name,
    n_channels,
    split,
    val_metrics,
    test_metrics,
    best_epoch=None,
    config_extra=None,
):
    row = {
        "protocol": protocol,
        "fold": fold,
        "model": model_name,
        "n_channels": n_channels,
        "channel_config_hash": channel_config_hash(n_channels),
        "train_subjects": ",".join(map(str, split["train"])),
        "val_subjects": ",".join(map(str, split["val"])),
        "test_subjects": ",".join(map(str, split["test"])),
        "sfreq": float(fs),
        "time_start_s": 1.0,
        "time_stop_s": 3.5,
        "seed": SEED,
        "best_epoch": best_epoch,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }
    if config_extra:
        row.update(config_extra)
    EXPERIMENT_ROWS[(protocol, fold, model_name, n_channels)] = row
    return row


# Rebuild explicit channel geometry because `epochs` was overwritten
# while loading all sessions.
experiment_info = mne.create_info(
    ch_names=channel_names.tolist(),
    sfreq=float(fs),
    ch_types="eeg",
)
experiment_info.set_montage(
    mne.channels.make_standard_montage("biosemi128"),
    match_case=False,
    on_missing="raise",
)

assert X.ndim == 3 and X.shape[1] == 128
assert len(X) == len(y) == len(subjects)
assert np.isfinite(X).all()
assert set(CHANNEL_COUNTS) == set(channel_configs_idx)
assert all(len(np.unique(channel_configs_idx[n])) == n for n in CHANNEL_COUNTS)
assert set(channel_configs_idx[16]).issubset(channel_configs_idx[32])
assert set(channel_configs_idx[32]).issubset(channel_configs_idx[64])
assert set(channel_configs_idx[64]).issubset(channel_configs_idx[128])
validate_split(DEV_SPLIT)

set_all_seeds()
print("Device:", device)
print("Development split:", DEV_SPLIT)
print("Channel hashes:", {n: channel_config_hash(n) for n in CHANNEL_COUNTS})


## Pipeline 1 — Welch bandpower + SVM


In [ ]:
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}


def extract_bandpower_tensor(data, sfreq):
    """Return float32 features with shape [trial, channel, band]."""
    features = np.empty(
        (len(data), data.shape[1], len(BANDS)),
        dtype=np.float32,
    )
    for trial_idx, trial in enumerate(
        tqdm(data, desc="Welch bandpower", leave=False)
    ):
        freqs, psd = welch(
            trial,
            fs=sfreq,
            nperseg=min(256, trial.shape[-1]),
            axis=-1,
        )
        for band_idx, (low, high) in enumerate(BANDS.values()):
            frequency_mask = (freqs >= low) & (freqs < high)
            features[trial_idx, :, band_idx] = np.log(
                psd[:, frequency_mask].mean(axis=-1) + 1e-12
            )
    return features


bandpower_cache = RESULTS_DIR / (
    f"bandpower_{X.shape[0]}x{X.shape[1]}x{X.shape[2]}_{float(fs):g}hz.npy"
)

if bandpower_cache.exists():
    bandpower_all = np.load(bandpower_cache, mmap_mode="r")
    expected_shape = (len(X), X.shape[1], len(BANDS))
    if bandpower_all.shape != expected_shape:
        raise ValueError(
            f"Stale bandpower cache: {bandpower_all.shape} != {expected_shape}"
        )
else:
    bandpower_all = extract_bandpower_tensor(X, fs)
    np.save(bandpower_cache, bandpower_all)


def run_svm_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
):
    masks = split_masks(split)
    idx = np.asarray(channel_configs_idx[n_channels], dtype=int)
    features = np.asarray(bandpower_all[:, idx, :]).reshape(len(X), -1)

    estimator = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale")),
    ])
    estimator.fit(features[masks["train"]], y[masks["train"]])

    val_pred = estimator.predict(features[masks["val"]])
    test_pred = estimator.predict(features[masks["test"]])
    val_metrics = classification_metrics(y[masks["val"]], val_pred)
    test_metrics = classification_metrics(y[masks["test"]], test_pred)

    store_predictions(
        protocol, fold, "SVM_bandpower", n_channels,
        "val", masks["val"], val_pred,
    )
    store_predictions(
        protocol, fold, "SVM_bandpower", n_channels,
        "test", masks["test"], test_pred,
    )
    row = store_result(
        protocol,
        fold,
        "SVM_bandpower",
        n_channels,
        split,
        val_metrics,
        test_metrics,
        config_extra={
            "representation": "Welch log-bandpower",
            "normalization": "StandardScaler fit on train only",
            "checkpoint": None,
        },
    )
    print(
        f"SVM | {n_channels:3d} ch | "
        f"val acc={val_metrics['accuracy']:.3f} | "
        f"test acc={test_metrics['accuracy']:.3f}"
    )
    return estimator, row


In [ ]:
svm_models = {}
for n_channels in CHANNEL_COUNTS:
    svm_models[n_channels], _ = run_svm_experiment(n_channels)


## Shared PyTorch training and evaluation

Normalization statistics are calculated from training subjects only. Every model/channel condition is reinitialized and gets its own best-validation checkpoint.


In [ ]:
class EEGArrayDataset(Dataset):
    def __init__(self, data, labels):
        self.X = torch.from_numpy(np.ascontiguousarray(data, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(labels, dtype=np.int64))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


def prepare_raw_splits(n_channels, split, target_sfreq=None):
    masks = split_masks(split)
    idx = np.asarray(channel_configs_idx[n_channels], dtype=int)
    data = np.asarray(X[:, idx, :], dtype=np.float32)

    actual_sfreq = float(fs)
    if target_sfreq is not None and not np.isclose(actual_sfreq, target_sfreq):
        ratio = Fraction(target_sfreq / actual_sfreq).limit_denominator(1000)
        data = resample_poly(
            data,
            up=ratio.numerator,
            down=ratio.denominator,
            axis=-1,
        ).astype(np.float32, copy=False)
        actual_sfreq = float(target_sfreq)

    train_data = data[masks["train"]]
    mean = train_data.mean(axis=(0, 2), keepdims=True, dtype=np.float64)
    std = train_data.std(axis=(0, 2), keepdims=True, dtype=np.float64)
    std = np.maximum(std, 1e-8)

    arrays = {
        name: ((data[mask] - mean) / std).astype(np.float32)
        for name, mask in masks.items()
    }
    labels = {name: y[mask] for name, mask in masks.items()}
    return arrays, labels, masks, actual_sfreq


def make_loaders(arrays, labels, batch_size):
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(
            EEGArrayDataset(arrays["train"], labels["train"]),
            batch_size=batch_size,
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
        "val": DataLoader(
            EEGArrayDataset(arrays["val"], labels["val"]),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
        "test": DataLoader(
            EEGArrayDataset(arrays["test"], labels["test"]),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
    }


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(batch_y)
        total_correct += (logits.argmax(1) == batch_y).sum().item()
        total_examples += len(batch_y)
    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def predict_torch(model, loader, criterion):
    model.eval()
    losses = 0.0
    labels = []
    predictions = []
    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        logits = model(batch_X)
        losses += criterion(logits, batch_y).item() * len(batch_y)
        labels.append(batch_y.cpu().numpy())
        predictions.append(logits.argmax(1).cpu().numpy())
    labels = np.concatenate(labels)
    predictions = np.concatenate(predictions)
    return losses / len(labels), labels, predictions


def fit_torch_stage(
    model,
    loaders,
    optimizer,
    epochs,
    patience,
    checkpoint_path,
    description,
):
    criterion = nn.CrossEntropyLoss()
    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None
    history = []
    epochs_without_improvement = 0

    progress = tqdm(range(1, epochs + 1), desc=description)
    for epoch in progress:
        train_loss, train_accuracy = train_one_epoch(
            model, loaders["train"], optimizer, criterion
        )
        val_loss, val_y, val_pred = predict_torch(
            model, loaders["val"], criterion
        )
        val_metrics = classification_metrics(val_y, val_pred)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        if val_metrics["accuracy"] > best_val_accuracy + 1e-12:
            best_val_accuracy = val_metrics["accuracy"]
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        progress.set_postfix(
            train=f"{train_accuracy:.3f}",
            val=f"{val_metrics['accuracy']:.3f}",
            best=f"{best_val_accuracy:.3f}",
        )
        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("Training ended without a checkpoint.")
    model.load_state_dict(best_state)
    torch.save(best_state, checkpoint_path)
    return pd.DataFrame(history), best_epoch


def finalize_torch_experiment(
    model,
    loaders,
    masks,
    split,
    protocol,
    fold,
    model_name,
    n_channels,
    best_epoch,
    checkpoint_path,
    config_extra,
):
    criterion = nn.CrossEntropyLoss()
    _, val_y, val_pred = predict_torch(model, loaders["val"], criterion)
    _, test_y, test_pred = predict_torch(model, loaders["test"], criterion)
    val_metrics = classification_metrics(val_y, val_pred)
    test_metrics = classification_metrics(test_y, test_pred)

    store_predictions(
        protocol, fold, model_name, n_channels,
        "val", masks["val"], val_pred,
    )
    store_predictions(
        protocol, fold, model_name, n_channels,
        "test", masks["test"], test_pred,
    )
    row = store_result(
        protocol,
        fold,
        model_name,
        n_channels,
        split,
        val_metrics,
        test_metrics,
        best_epoch=best_epoch,
        config_extra={
            **config_extra,
            "checkpoint": str(checkpoint_path),
        },
    )
    print(
        f"{model_name} | {n_channels:3d} ch | "
        f"val acc={val_metrics['accuracy']:.3f} | "
        f"test acc={test_metrics['accuracy']:.3f}"
    )
    return row


## Pipeline 2 — EEGNet trained from scratch


In [ ]:
def run_eegnet_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
    epochs=50,
    patience=10,
    batch_size=32,
):
    set_all_seeds()
    arrays, labels, masks, actual_sfreq = prepare_raw_splits(
        n_channels, split
    )
    loaders = make_loaders(arrays, labels, batch_size=batch_size)
    model = EEGNet(n_channels=n_channels, n_classes=4).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    checkpoint_path = CHECKPOINT_DIR / (
        f"{protocol}_{fold}_eegnet_{n_channels}ch_seed{SEED}.pt"
    )
    history, best_epoch = fit_torch_stage(
        model,
        loaders,
        optimizer,
        epochs=epochs,
        patience=patience,
        checkpoint_path=checkpoint_path,
        description=f"EEGNet {n_channels}ch",
    )
    history.assign(
        protocol=protocol,
        fold=fold,
        model="EEGNet_scratch",
        n_channels=n_channels,
    ).to_csv(
        RESULTS_DIR / f"{protocol}_{fold}_eegnet_{n_channels}ch_history.csv",
        index=False,
    )
    row = finalize_torch_experiment(
        model,
        loaders,
        masks,
        split,
        protocol,
        fold,
        "EEGNet_scratch",
        n_channels,
        best_epoch,
        checkpoint_path,
        config_extra={
            "representation": "raw time-domain EEG",
            "normalization": "per-channel z-score fit on train only",
            "model_sfreq": actual_sfreq,
            "epochs_requested": epochs,
            "batch_size": batch_size,
            "learning_rate": 1e-3,
        },
    )
    return model, row


In [ ]:
eegnet_models = {}
for n_channels in CHANNEL_COUNTS:
    eegnet_models[n_channels], _ = run_eegnet_experiment(n_channels)


## Pipeline 3 — pretrained InterpolatedBIOT

This pipeline is intentionally opt-in because the current notebook has Kaggle Internet disabled. Enable Internet (or attach the cached package and Hugging Face checkpoint), install `braindecode[hug]`, restart the kernel, and set `RUN_BIOT = True`.

The training schedule first fits only a fresh four-class head, then fine-tunes the pretrained encoder with a lower learning rate. The geometry-derived interpolation stays frozen in both stages.


In [ ]:
BIOT_REPO_ID = "braindecode/biot-pretrained-six-datasets-18chs"


def load_interpolated_biot(n_channels, n_times):
    try:
        import braindecode
        from braindecode.models import InterpolatedBIOT
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            'InterpolatedBIOT is unavailable. Enable Kaggle Internet, run '
            '`%pip install -q "braindecode[hug]>=1.5,<2"`, then restart '
            "the kernel. Alternatively attach the package/checkpoint as "
            "a Kaggle input."
        ) from exc

    idx = np.asarray(channel_configs_idx[n_channels], dtype=int)
    subset_info = mne.pick_info(experiment_info.copy(), idx.tolist())
    chs_info = subset_info["chs"]

    try:
        model = InterpolatedBIOT.from_pretrained(
            BIOT_REPO_ID,
            chs_info=chs_info,
            n_times=n_times,
            sfreq=200,
            strict=False,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Could not load real pretrained weights from {BIOT_REPO_ID}. "
            "This experiment will not fall back to random weights. Enable "
            "Internet or attach a Hugging Face cache containing the model."
        ) from exc

    model.reset_head(4)
    projection_shape = tuple(model.interpolation_layer.matrix.shape)
    if projection_shape[1] != n_channels:
        raise RuntimeError(
            f"Unexpected BIOT interpolation shape {projection_shape}; "
            f"expected source dimension {n_channels}."
        )
    return model, braindecode.__version__, projection_shape


def run_biot_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
    head_epochs=10,
    finetune_epochs=30,
    patience=8,
    batch_size=16,
):
    set_all_seeds()
    arrays, labels, masks, actual_sfreq = prepare_raw_splits(
        n_channels, split, target_sfreq=200
    )
    loaders = make_loaders(arrays, labels, batch_size=batch_size)
    model, braindecode_version, projection_shape = load_interpolated_biot(
        n_channels, n_times=arrays["train"].shape[-1]
    )
    model = model.to(device)

    # Stage 1: linear probe. Only the new four-class head is trainable.
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.final_layer.parameters():
        parameter.requires_grad = True

    head_checkpoint = CHECKPOINT_DIR / (
        f"{protocol}_{fold}_biot_{n_channels}ch_head_seed{SEED}.pt"
    )
    head_optimizer = torch.optim.AdamW(
        model.final_layer.parameters(), lr=1e-3, weight_decay=1e-4
    )
    head_history, head_best_epoch = fit_torch_stage(
        model,
        loaders,
        head_optimizer,
        epochs=head_epochs,
        patience=min(patience, head_epochs),
        checkpoint_path=head_checkpoint,
        description=f"BIOT head {n_channels}ch",
    )

    # Stage 2: fine-tune the encoder, keeping interpolation fixed.
    for parameter in model.parameters():
        parameter.requires_grad = True
    for parameter in model.interpolation_layer.parameters():
        parameter.requires_grad = False

    finetune_checkpoint = CHECKPOINT_DIR / (
        f"{protocol}_{fold}_biot_{n_channels}ch_finetuned_seed{SEED}.pt"
    )
    finetune_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-4,
        weight_decay=1e-4,
    )
    finetune_history, finetune_best_epoch = fit_torch_stage(
        model,
        loaders,
        finetune_optimizer,
        epochs=finetune_epochs,
        patience=patience,
        checkpoint_path=finetune_checkpoint,
        description=f"BIOT tune {n_channels}ch",
    )

    history = pd.concat([
        head_history.assign(stage="linear_probe"),
        finetune_history.assign(stage="fine_tune"),
    ], ignore_index=True)
    history.assign(
        protocol=protocol,
        fold=fold,
        model="BIOT_pretrained",
        n_channels=n_channels,
    ).to_csv(
        RESULTS_DIR / f"{protocol}_{fold}_biot_{n_channels}ch_history.csv",
        index=False,
    )

    row = finalize_torch_experiment(
        model,
        loaders,
        masks,
        split,
        protocol,
        fold,
        "BIOT_pretrained",
        n_channels,
        finetune_best_epoch,
        finetune_checkpoint,
        config_extra={
            "representation": "raw EEG resampled to 200 Hz",
            "normalization": "per-channel z-score fit on train only",
            "model_sfreq": actual_sfreq,
            "pretrained_repo": BIOT_REPO_ID,
            "braindecode_version": braindecode_version,
            "interpolation_shape": str(projection_shape),
            "interpolation_trainable": False,
            "head_best_epoch": head_best_epoch,
            "head_epochs_requested": head_epochs,
            "finetune_epochs_requested": finetune_epochs,
            "batch_size": batch_size,
            "head_learning_rate": 1e-3,
            "finetune_learning_rate": 1e-4,
        },
    )
    return model, row


In [ ]:
RUN_BIOT = False  # Set True after the package/checkpoint setup above.

biot_models = {}
if RUN_BIOT:
    for n_channels in CHANNEL_COUNTS:
        biot_models[n_channels], _ = run_biot_experiment(n_channels)
else:
    print(
        "BIOT not run. Complete the setup in the previous markdown cell, "
        "then set RUN_BIOT = True."
    )


## Results, retention, subject metrics, and optional LOSO

The main table contains one row per trained model/channel condition. Retention is always computed against the same model's 128-channel result in the same protocol/fold.

For final LOSO, one held-out subject is the test set and a different held-out subject is the validation set. This leaves eight training subjects and prevents using the test subject for early stopping. LOSO is defined but not launched automatically because all three tracks across 10 folds require 120 complete training runs.


In [ ]:
def build_results_tables():
    results = pd.DataFrame(EXPERIMENT_ROWS.values())
    predictions = (
        pd.concat(PREDICTION_FRAMES.values(), ignore_index=True)
        if PREDICTION_FRAMES
        else pd.DataFrame()
    )
    if results.empty:
        return results, predictions, pd.DataFrame()

    results = results.sort_values(
        ["protocol", "fold", "model", "n_channels"],
        ascending=[True, True, True, False],
    ).reset_index(drop=True)

    full_channel_accuracy = (
        results[results["n_channels"] == 128]
        .set_index(["protocol", "fold", "model"])["test_accuracy"]
    )
    results["accuracy_retention_percent"] = [
        100.0 * row.test_accuracy /
        full_channel_accuracy.get(
            (row.protocol, row.fold, row.model), np.nan
        )
        for row in results.itertuples()
    ]

    if predictions.empty:
        subject_metrics = pd.DataFrame()
    else:
        subject_metrics = (
            predictions[predictions["partition"] == "test"]
            .groupby(
                ["protocol", "fold", "model", "n_channels", "subject"],
                as_index=False,
            )
            .apply(
                lambda frame: pd.Series(
                    classification_metrics(frame["y_true"], frame["y_pred"])
                ),
            )
            .reset_index(drop=True)
        )

    results.to_csv(RESULTS_DIR / "channel_reduction_results.csv", index=False)
    predictions.to_csv(RESULTS_DIR / "trial_predictions.csv", index=False)
    subject_metrics.to_csv(
        RESULTS_DIR / "subject_level_metrics.csv", index=False
    )
    return results, predictions, subject_metrics


results_df, predictions_df, subject_metrics_df = build_results_tables()
display(results_df)
display(subject_metrics_df)


def plot_development_confusions(predictions=predictions_df):
    if predictions.empty:
        print("No predictions are available yet.")
        return

    data = predictions[
        (predictions["protocol"] == "development") &
        (predictions["fold"] == "fixed") &
        (predictions["partition"] == "test")
    ]
    for model_name in data["model"].unique():
        fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
        for ax, n_channels in zip(axes, CHANNEL_COUNTS):
            subset = data[
                (data["model"] == model_name) &
                (data["n_channels"] == n_channels)
            ]
            matrix = confusion_matrix(
                subset["y_true"],
                subset["y_pred"],
                labels=range(4),
                normalize="true",
            )
            image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
            ax.set_title(f"{n_channels} channels")
            ax.set_xticks(range(4), CLASS_NAMES.values(), rotation=45)
            ax.set_yticks(range(4), CLASS_NAMES.values())
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
        fig.colorbar(image, ax=axes, shrink=0.75)
        fig.suptitle(f"{model_name}: normalized test confusion matrices")
        plt.show()


plot_development_confusions()


In [ ]:
def make_loso_splits():
    """Deterministic nested validation: next subject validates each test fold."""
    unique_subjects = sorted(np.unique(subjects).tolist())
    folds = []
    for position, test_subject in enumerate(unique_subjects):
        val_subject = unique_subjects[(position + 1) % len(unique_subjects)]
        train_subjects = tuple(
            subject
            for subject in unique_subjects
            if subject not in (test_subject, val_subject)
        )
        folds.append({
            "fold": f"test_subject_{test_subject:02d}",
            "split": {
                "train": train_subjects,
                "val": (val_subject,),
                "test": (test_subject,),
            },
        })
    return folds


def run_loso(pipelines=("svm",), channel_counts=CHANNEL_COUNTS):
    """
    Run official subject-aware folds.

    Start with ("svm",) to verify the protocol cheaply. Running all three
    pipelines over every count launches 120 model fits.
    """
    allowed = {"svm", "eegnet", "biot"}
    unknown = set(pipelines) - allowed
    if unknown:
        raise ValueError(f"Unknown pipelines: {sorted(unknown)}")

    for fold_spec in make_loso_splits():
        fold = fold_spec["fold"]
        split = fold_spec["split"]
        print(f"\n===== {fold}: {split} =====")
        for n_channels in channel_counts:
            if "svm" in pipelines:
                run_svm_experiment(
                    n_channels, split, protocol="loso", fold=fold
                )
            if "eegnet" in pipelines:
                run_eegnet_experiment(
                    n_channels, split, protocol="loso", fold=fold
                )
            if "biot" in pipelines:
                run_biot_experiment(
                    n_channels, split, protocol="loso", fold=fold
                )
        build_results_tables()


# Recommended progression:
# run_loso(pipelines=("svm",))
# run_loso(pipelines=("eegnet",))
# run_loso(pipelines=("biot",))
#
# Then refresh and aggregate:
# results_df, predictions_df, subject_metrics_df = build_results_tables()
# loso_summary = (
#     results_df[results_df["protocol"] == "loso"]
#     .groupby(["model", "n_channels"])
#     [["test_accuracy", "test_macro_f1", "test_balanced_accuracy"]]
#     .agg(["mean", "std"])
# )
# display(loso_summary)
